# 🤖 Minion AI — Training Notebook

Modernized GPT training stack.  Train on Colab → run locally on RTX 3050.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 or better)
2. Run cells top to bottom
3. Checkpoints save to Google Drive automatically
4. If session disconnects, **re-run all cells** — training resumes from the last checkpoint

---
**Mode A** (pretrain): Train Minion from scratch (~125M params, ~3-5 sessions)

**Mode B** (qlora_finetune): Fine-tune GPT-2 XL with LoRA adapters (~1 session)

## Cell 1 — Mount Google Drive
Checkpoints will be saved here and survive session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CKPT_DIR = '/content/drive/MyDrive/minion_ckpts'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Checkpoint directory: {DRIVE_CKPT_DIR}')
print('Existing checkpoints:', os.listdir(DRIVE_CKPT_DIR) or 'none yet')

## Cell 2 — Install dependencies

In [ ]:
# Install pinned requirements (train side)
%pip install -q \
    torch>=2.2.0 \
    bitsandbytes>=0.43.0 \
    peft>=0.10.0 \
    transformers>=4.40.0 \
    accelerate>=0.28.0 \
    tokenizers>=0.19.0 \
    datasets>=2.18.0 \
    PyYAML>=6.0 \
    wandb>=0.16.0 \
    tqdm>=4.66.0 \
    sentencepiece>=0.2.0

print('Dependencies installed ✓')

## Cell 3 — Clone / pull Minion AI repo

In [ ]:
import os, sys

# ── CONFIGURATION ──────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/mlwithharsh/Minion.git'
REPO_DIR = '/content/Minion'   # git clones into a folder named after the repo
# ───────────────────────────────────────────────────────────────────────────

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest changes...')
    os.system(f'git -C {REPO_DIR} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

# Set working directory AND sys.path so all cells can import from the repo
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Working directory: {os.getcwd()}')
print('Key files present:', [f for f in ['train.py', 'model.py', 'data.py', 'tokenizer/tokenizer.json'] if os.path.exists(f)])

## Cell 4 — Detect GPU
Minion AI auto-detects GPU properties and adjusts precision accordingly.
T4 → fp16 + GradScaler | A100 → bfloat16

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1024**3
    print(f'GPU:        {props.name}')
    print(f'VRAM:       {vram_gb:.1f} GB')
    print(f'Compute:    {props.major}.{props.minor}')
    print(f'bf16:       {torch.cuda.is_bf16_supported()}')
    print()
    if vram_gb < 12:
        print('⚠️  Less than 12 GB VRAM — reduce batch_size or block_size if OOM.')
    elif vram_gb >= 40:
        print('🚀 A100/H100 — you can scale to 350M+ params!')
        print('   Edit config: n_layer=24, n_head=16, n_embd=1024')
else:
    print('❌ No GPU — change Runtime → GPU!')

## Cell 5 — Tokenizer check
The 32k BPE tokenizer is bundled in the repo (`tokenizer/tokenizer.json`).
This cell just verifies it loaded — **no training needed**.

In [ ]:
import os
from tokenizers import Tokenizer

tok_path = 'tokenizer/tokenizer.json'
if os.path.exists(tok_path):
    tok = Tokenizer.from_file(tok_path)
    print(f'✓ Tokenizer ready: vocab_size={tok.get_vocab_size()}')
    sample = tok.encode('The future of AI is here').ids
    print(f'  Sample encode: {sample[:8]}...')
else:
    print('Tokenizer not found — training now (takes ~10 min)...')
    os.system('python tokenizer_train.py --mode train '
              '--hf_dataset HuggingFaceFW/fineweb-edu '
              '--hf_dataset_config sample-10BT '
              '--vocab_size 32768 --algorithm bpe '
              '--tokenizer_dir tokenizer --max_docs 100000')
    print('Tokenizer training complete ✓')

## Cell 6 — Configure training
Edit the config path and any overrides here.

In [ ]:
import os

# Drive checkpoint dir (set in Cell 1)
DRIVE_CKPT_DIR = '/content/drive/MyDrive/minion_ckpts'

# Choose config:
#   Mode A pretrain (125M, T4):       'config/minion_pretrain_colab.yaml'
#   Mode B qlora finetune (GPT-2 XL): 'config/minion_qlora_gpt2xl_colab.yaml'
#   Smoke test (< 2 min on CPU):      'config/smoke_test.yaml'
CONFIG_PATH = 'config/minion_pretrain_colab.yaml'

OVERRIDES = {
    'ckpt_dir': DRIVE_CKPT_DIR,   # Always point to Drive
    # Uncomment to override config values:
    # 'wandb_log':   True,
    # 'max_iters':   5000,
    # 'batch_size':  4,            # Lower if OOM
    # 'compile':     False,        # Disable torch.compile for faster startup
    # 'n_layer':     24,           # Larger model (A100 only)
    # 'n_embd':      1024,
}

print(f'Config:    {CONFIG_PATH}')
print(f'Ckpt dir:  {DRIVE_CKPT_DIR}')
existing = [f for f in os.listdir(DRIVE_CKPT_DIR)] if os.path.exists(DRIVE_CKPT_DIR) else []
print(f'Drive ckpts: {existing or "none — fresh run"}')

## Cell 7 — Start training

This cell is **self-contained** — it re-establishes the repo path on every run.
Safe to re-run after a Colab disconnect (training auto-resumes from Drive checkpoint).

What to expect:
```
GPU: Tesla T4  |  VRAM: 15.8 GB  |  Compute: 7.5
bf16 supported: False
Precision: float16  |  AMP: True
Minion: 124.44M parameters
Optimizer: 8-bit AdamW — ~50% optimizer memory saving
step      0: train 4.xx  val 4.xx  vram 11.xGB
  checkpoint saved: iter=500, val_loss=x.xxxx  ← Drive
```

In [ ]:
import os, sys

# ── Re-establish repo path (safe to re-run after disconnect) ──────────────
REPO_DIR = '/content/Minion'
if not os.path.exists(os.path.join(REPO_DIR, 'train.py')):
    # Repo not present — clone it
    print('Repo not found — cloning...')
    os.system('git clone https://github.com/mlwithharsh/Minion.git ' + REPO_DIR)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'Repo dir:  {os.getcwd()}')

# ── Re-establish Drive checkpoint dir ─────────────────────────────────────
DRIVE_CKPT_DIR = '/content/drive/MyDrive/minion_ckpts'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

# ── Re-establish config (edit here if you skipped Cell 6) ─────────────────
CONFIG_PATH = 'config/minion_pretrain_colab.yaml'
OVERRIDES   = {'ckpt_dir': DRIVE_CKPT_DIR}
# ──────────────────────────────────────────────────────────────────────────

from train import train
train(CONFIG_PATH, **OVERRIDES)

## Cell 8 — Quick generation test (after training)
Test the model directly in Colab before downloading to local machine.

In [ ]:
import os, sys, torch

REPO_DIR = '/content/Minion'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

PROMPT         = 'The future of artificial intelligence is'  # ← edit
DRIVE_CKPT_DIR = '/content/drive/MyDrive/minion_ckpts'
CKPT           = os.path.join(DRIVE_CKPT_DIR, 'ckpt.pt')

if not os.path.exists(CKPT):
    print(f'No checkpoint at {CKPT} — run Cell 7 first')
else:
    from inference.quantize import load_quantized_model
    from inference.generate import generate_text

    model, tokenizer, cfg = load_quantized_model(
        ckpt_path=CKPT, quantize=True, device='cuda'
    )
    text = generate_text(
        model=model, tokenizer=tokenizer, cfg=cfg,
        prompt=PROMPT, max_new_tokens=200, temperature=0.8,
        top_k=50, device='cuda',
    )
    print('=== Generated text ===')
    print(PROMPT + text)

## Cell 9 — Download checkpoint to local machine

**Option A** (recommended): rclone on your local machine:
```bash
python inference/sync_checkpoint.py --method rclone --remote "gdrive:minion_ckpts" --local_dir ./checkpoints
```

**Option B**: Download directly from this cell:

In [ ]:
from google.colab import files
import os

DRIVE_CKPT_DIR = '/content/drive/MyDrive/minion_ckpts'
CKPT = os.path.join(DRIVE_CKPT_DIR, 'ckpt.pt')

if os.path.exists(CKPT):
    size_mb = os.path.getsize(CKPT) / 1024**2
    print(f'Downloading ckpt.pt ({size_mb:.0f} MB) ...')
    files.download(CKPT)
else:
    print(f'No checkpoint at {CKPT} — run Cell 7 first')